In [ ]:
# Day 48 - Production Level Churn Prediction
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, roc_curve, auc
from xgboost import XGBClassifier

# Dataset
n = 1000

df = pl.DataFrame({
    "tenure": np.random.randint(1, 72, n),
    "monthly_charges": np.random.uniform(20, 120, n),
    "total_charges": np.random.uniform(100, 8000, n),
    "contract": np.random.choice(
        ["Month", "OneYear", "TwoYear"], n
    ),
    "churn": np.random.randint(0, 2, n)
})

# Feature Engineering
df = df.with_columns(
    (pl.col("total_charges") / pl.col("tenure"))
    .alias("avg_monthly_rev")
)

# Features / Target
X = df.drop("churn").to_pandas()
y = df["churn"].to_numpy()

num_cols = [
    "tenure",
    "monthly_charges",
    "total_charges",
    "avg_monthly_rev"
]

cat_cols = ["contract"]

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

# Pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", XGBClassifier(
        eval_metric="logloss",
        random_state=42
    ))
])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Hyperparameter Tuning
param_grid = {
    "clf__n_estimators": [50, 100],
    "clf__max_depth": [3, 5]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

# Save Model
os.makedirs("models", exist_ok=True)
joblib.dump(best_model, "models/churn_model.pkl")

# Prediction
preds = best_model.predict(X_test)
probs = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, preds))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, probs)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr,
         label=f"AUC = {auc(fpr,tpr):.2f}")
plt.plot([0,1],[0,1],"k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# Correlation Heatmap
numeric_df = df.select(num_cols + ["churn"])

plt.figure(figsize=(6,5))
sns.heatmap(
    numeric_df.to_pandas().corr(),
    annot=True,
    cmap="coolwarm"
)
plt.title("Feature Correlation")
plt.show()